In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-08-transfer-and-sim2real.py"

# 從模擬到實驗

用 FreeSolv 成對的實驗與計算水合自由能，在相同的測試分子上比較四種預測方式。程式在獨立的 Python 程序中執行。試做練習時，把程式加在分析格末端，再重跑分析格與顯示格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os

import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# FreeSolv，實驗與模擬水合自由能（kcal/mol）。Mobley & Guthrie, DOI 10.1007/s10822-014-9747-x
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR") or dc.utils.data_utils.get_data_dir())
tasks, (data,), _ = dc.molnet.load_sampl(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=str(cache))
assert tasks == ["expt"] and np.all(data.w[:, 0] > 0)
raw = pd.read_csv(cache / "SAMPL.csv")
pairs = pd.DataFrame({"smiles": data.ids, "measured": data.y[:, 0]}).merge(
    raw, on="smiles", validate="one_to_one", how="left")
assert len(pairs) == len(raw) and np.allclose(pairs.measured, pairs.expt)
assert np.isfinite(pairs[["measured", "calc"]]).all().all()
pairs["structure"] = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in pairs.smiles]
assert pairs.structure.is_unique
pairs = pairs.sort_values("structure").reset_index(drop=True)

def descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return [Descriptors.MolWt(mol), Crippen.MolLogP(mol), rdMolDescriptors.CalcTPSA(mol),
            Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol), Lipinski.NumRotatableBonds(mol)]

X = np.array([descriptors(s) for s in pairs.structure])
measured = pairs.measured.to_numpy()
calc = pairs.calc.to_numpy()
order = np.random.default_rng(42).permutation(len(pairs))
test, pool = order[:129], order[129:]
assert not set(test) & set(pool)
def fit_predict(train, train_labels):
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    return model.fit(X[train], train_labels).predict(X[test])

rows = []
for budget in [16, 64, 256]:
    train = pool[:budget]
    residual = measured[train] - calc[train]
    predictions = {
        "Structure only": fit_predict(train, measured[train]),
        "Simulation": calc[test],
        "Constant shift": calc[test] + residual.mean(),
        "Delta model": calc[test] + fit_predict(train, residual)}
    for method, pred in predictions.items():
        rows.append((budget, method, mean_absolute_error(measured[test], pred)))
scores = pd.DataFrame(rows, columns=["train", "method", "MAE"])
print("paired / test:", len(pairs), len(test))
print(scores.pivot(index="train", columns="method", values="MAE").round(3))

fig, ax = plt.subplots(figsize=(7.1, 4.4), layout="constrained")
styles = [("#4f6498", "o", "-"), ("#555555", "x", "--"),
          ("#977522", "s", ":"), ("#007369", "D", "-")]
for (method, group), (color, marker, line) in zip(scores.groupby("method", sort=False), styles):
    ax.plot(group.train, group.MAE, label=method, color=color, marker=marker, linestyle=line)
ax.set(xscale="log", xlabel="Training molecules with an experimental label",
       ylabel="Test MAE (kcal/mol)", xticks=[16, 64, 256], xticklabels=["16", "64", "256"])
ax.minorticks_off()
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
out = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "12-08-freesolv-transfer.png", dpi=180)

In [ ]:
from IPython.display import Image, display

import os
environment = dict(os.environ, VECLIB_MAXIMUM_THREADS="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MPLBACKEND="Agg")
subprocess.run([python, str(script)], check=True, env=environment)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-08-freesolv-transfer.png")))